# Laboratorio 3 - Integracao e Interoperabilidade de Sistemas

**Relatorio para entrega em PDF/notebook**  
Disciplina: Gestao de Sistemas de Informacao (GSI)  
Alunos: Lucas Garcia e Luis Augusto  
Tema: Middleware, ESB, API REST, interoperabilidade semantica e padronizacao ePING

## 1. Objetivo do laboratorio

O laboratorio simula a integracao de sistemas legados heterogeneos,
conhecidos como "ilhas de dados". A atividade mostra primeiro a
falha de interoperabilidade causada por contratos de dados
diferentes e, depois, implementa mecanismos para resolver o
problema: um middleware/ESB, uma API REST padronizada, um tradutor
semantico de unidades e um gateway de padronizacao inspirado na
ePING.

## 2. Preparacao do ambiente

Os scripts foram mantidos na pasta do laboratorio e importados pelo
notebook para que a entrega funcione como relatorio e tambem como
evidencia executavel. A API Flask nao precisa ficar rodando em
background para a demonstracao: usaremos o `test_client()` do Flask
para validar o contrato HTTP/JSON diretamente no notebook.

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display

POSSIVEIS_DIRS = [Path.cwd(), Path.cwd() / "GSI" / "lab3"]
BASE_DIR = next(path for path in POSSIVEIS_DIRS if (path / "legacy_systems.py").exists())

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print(f"Diretorio do laboratorio: {BASE_DIR}")
print("Arquivos principais:")
for caminho in sorted(BASE_DIR.glob("*.py")):
    print(f"- {caminho.name}")

Diretorio do laboratorio: /home/lucas/Documentos/ifes-4ano/GSI/lab3
Arquivos principais:
- gerar_notebook_lab3.py
- integrador_esb.py
- interop_semantica.py
- legacy_systems.py
- legacy_systems_original.py
- microservico_eping.py
- padronizacao_eping.py


## 3. Parte 1 - O desafio da fragmentacao

O Sistema de RH armazena o funcionario com os campos `id_func`,
`nome_completo` e `salario_base`. O Sistema Financeiro, porem,
espera receber `cod` e `valor`. Essa diferenca de contrato impede a
integracao direta, mesmo que os dados representem a mesma entidade
de negocio.

In [2]:
from legacy_systems_original import (
    funcionario_rh as funcionario_rh_original,
    processar_pagamento as processar_pagamento_original,
)

print("Dados vindos do RH:")
print(funcionario_rh_original)
print()
print("Tentativa de envio direto ao Financeiro:")
processar_pagamento_original(funcionario_rh_original)

Dados vindos do RH:
{'id_func': 101, 'nome_completo': 'Eduardo Amaral', 'salario_base': 5000.0}

Tentativa de envio direto ao Financeiro:
ERRO DE INTEROPERABILIDADE: Chave não encontrada 'cod'


A falha ocorre porque o Financeiro procura a chave `cod`, mas o RH
envia `id_func`. Isso caracteriza uma falha de interoperabilidade
tecnica e tambem semantica: os sistemas nao compartilham o mesmo
contrato de dados.

## 4. Parte 2 - Middleware e barramento ESB

Para resolver a incompatibilidade, foi implementado o
`IntegradorESB`. Ele atua como uma camada intermediaria que traduz o
contrato do RH para o contrato do Financeiro, preservando o
significado dos dados.

In [3]:
from integrador_esb import IntegradorESB
from legacy_systems import funcionario_rh, processar_pagamento

dados_convertidos = IntegradorESB.transformar_rh_para_financeiro(funcionario_rh)

print("Dados originais do RH:")
print(funcionario_rh)
print()
print("Dados convertidos pelo ESB:")
print(dados_convertidos)
print()
print("Envio ao Financeiro apos conversao:")
processar_pagamento(dados_convertidos)

assert dados_convertidos == {"cod": 101, "valor": 5000.00}

Dados originais do RH:
{'id_func': 101, 'nome_completo': 'Eduardo Amaral', 'salario_base': 5000.0}

Dados convertidos pelo ESB:
{'cod': 101, 'valor': 5000.0}

Envio ao Financeiro apos conversao:
Processando ID 101: Valor R$ 5000.0


Com o ESB, a integracao deixa de depender de alteracoes diretas nos
sistemas legados. O RH continua publicando seu formato original e o
Financeiro continua consumindo o formato que espera, enquanto o
middleware centraliza a transformacao.

## 5. Parte 3 - API REST e microservico ePING

O roteiro tambem pede a simulacao de um microservico governamental
seguindo um contrato padronizado. A implementacao usa Flask e expõe
o endpoint `/api/cidadao/<id>`, retornando JSON com `cpf`, `nome` e
`status`.

In [4]:
from microservico_eping import app

respostas = []
with app.test_client() as client:
    for cidadao_id in [1, 2, 999]:
        resposta = client.get(f"/api/cidadao/{cidadao_id}")
        respostas.append({
            "endpoint": f"/api/cidadao/{cidadao_id}",
            "status_http": resposta.status_code,
            "json": resposta.get_json(),
        })

pd.DataFrame(respostas)

,endpoint,status_http,json
0,/api/cidadao/1,200,"{'cpf': '123.456.789-00', 'nome': 'Eduardo Ama..."
1,/api/cidadao/2,200,"{'cpf': '987.654.321-00', 'nome': 'Maria Silva..."
2,/api/cidadao/999,200,{'erro': 'Não encontrado'}


O uso de HTTP e JSON resolve a interoperabilidade tecnica. O
contrato do JSON, com campos padronizados, apoia a
interoperabilidade semantica, pois consumidores diferentes passam a
interpretar os dados da mesma forma.

## 6. Parte 4 - Interoperabilidade semantica

No desafio de temperatura, enviar apenas o numero `100` e omitir a
unidade pode gerar erro grave de interpretacao: `100 C` e `100 F`
significam coisas diferentes. A funcao `tradutor_universal()`
padroniza o dado para Celsius antes do envio ao Sistema de Saude.

In [5]:
from interop_semantica import enviar_para_sistema_saude, tradutor_universal

cenarios = [(37, "C"), (100, "F")]
resultados = []

for valor, unidade in cenarios:
    valor_convertido, unidade_convertida = tradutor_universal(valor, unidade)
    enviar_para_sistema_saude(valor_convertido, unidade_convertida)
    resultados.append({
        "valor_original": valor,
        "unidade_original": unidade,
        "valor_enviado": round(valor_convertido, 2),
        "unidade_enviada": unidade_convertida,
    })

assert resultados[0]["valor_enviado"] == 37
assert resultados[1]["valor_enviado"] == 37.78

pd.DataFrame(resultados)


Dado recebido: 37°C
Nenhuma conversão necessária.

Enviando para Sistema de Saúde...
Temperatura registrada: 37.00°C
Interoperabilidade semântica garantida.


Dado recebido: 100°F
Convertendo de Fahrenheit para Celsius...
Valor convertido: 37.78°C

Enviando para Sistema de Saúde...
Temperatura registrada: 37.78°C
Interoperabilidade semântica garantida.



,valor_original,unidade_original,valor_enviado,unidade_enviada
0,37,C,37.00,C
1,100,F,37.78,C


**Reflexao:** apenas transmitir o numero sem a unidade fere a
interoperabilidade semantica porque o receptor recebe sintaxe, mas
nao recebe significado. A unidade faz parte do contrato do dado.
Sem ela, dois sistemas podem aceitar a mensagem e ainda assim tomar
decisoes erradas.

## 7. Parte 5 - Governanca e padronizacao ePING

O ultimo desafio simula a integracao entre orgaos publicos. O
Ministerio da Fazenda envia CPF com mascara e nome em formato livre,
enquanto o sistema consumidor exige CPF numerico e nome em
maiusculas. A funcao `normalizar_cadastro()` atua como gateway de
integracao.

In [6]:
from padronizacao_eping import normalizar_cadastro

cadastros_origem = [
    {"cpf": "123.456.789-00", "nome": "Eduardo Amaral"},
    {"cpf": "987.654.321-00", "nome": "Maria Silva"},
]

cadastros_normalizados = [
    normalizar_cadastro(cadastro) for cadastro in cadastros_origem
]

assert cadastros_normalizados[0] == {
    "cpf": "12345678900",
    "nome": "EDUARDO AMARAL",
    "origem_validada": True,
}

pd.DataFrame(cadastros_normalizados)


DADOS ORIGINAIS RECEBIDOS
{'cpf': '123.456.789-00', 'nome': 'Eduardo Amaral'}

DADOS PADRONIZADOS (CONTRATO ePING)
{'cpf': '12345678900', 'nome': 'EDUARDO AMARAL', 'origem_validada': True}

DADOS ORIGINAIS RECEBIDOS
{'cpf': '987.654.321-00', 'nome': 'Maria Silva'}

DADOS PADRONIZADOS (CONTRATO ePING)
{'cpf': '98765432100', 'nome': 'MARIA SILVA', 'origem_validada': True}


,cpf,nome,origem_validada
0,12345678900,EDUARDO AMARAL,True
1,98765432100,MARIA SILVA,True


**Reflexao:** se o formato do CPF mudar, a atualizacao deve ser
concentrada no middleware/gateway de integracao sempre que possivel.
Essa decisao reduz acoplamento, evita alteracoes duplicadas nos
ministerios consumidores e torna a arquitetura mais resiliente a
mudancas de contrato.

## 8. Codigo-fonte desenvolvido

Para deixar a entrega autocontida, os principais scripts do
laboratorio sao exibidos abaixo em formato textual.

In [7]:
def mostrar_codigo_fonte(nome_arquivo: str) -> None:
    caminho = BASE_DIR / nome_arquivo
    codigo = caminho.read_text(encoding="utf-8")
    display(Markdown(f"### Codigo-fonte: `{nome_arquivo}`"))
    display(Markdown(f"```python\n{codigo}\n```"))


for arquivo in [
    "legacy_systems_original.py",
    "legacy_systems.py",
    "integrador_esb.py",
    "microservico_eping.py",
    "interop_semantica.py",
    "padronizacao_eping.py",
]:
    mostrar_codigo_fonte(arquivo)

### Codigo-fonte: `legacy_systems_original.py`

```python
# ============================================================
# LABORATÓRIO 3 - PARTE 1
# DESAFIO DAS ILHAS DE DADOS
# ============================================================

"""
Este script demonstra um problema clássico de integração:
Dois sistemas legados que utilizam estruturas de dados diferentes
e não conseguem se comunicar diretamente.

Aqui ocorre um problema de INTEROPERABILIDADE TÉCNICA,
pois os formatos das chaves são incompatíveis.
"""

# ------------------------------------------------------------
# Sistema A: Ilha de Dados de RH
# ------------------------------------------------------------
# Estrutura utilizada pelo setor de Recursos Humanos
funcionario_rh = {
    "id_func": 101,
    "nome_completo": "Eduardo Amaral",
    "salario_base": 5000.00
}


# ------------------------------------------------------------
# Sistema B: Ilha de Dados Financeira
# ------------------------------------------------------------
# Este sistema espera campos com nomes diferentes:
# 'cod' e 'valor'
def processar_pagamento(dados_pagamento):
    """
    Sistema financeiro que processa pagamentos.
    Ele falha se as chaves esperadas não existirem.
    """
    try:
        print(f"Processando ID {dados_pagamento['cod']}: Valor R$ {dados_pagamento['valor']}")
    except KeyError as e:
        print(f"ERRO DE INTEROPERABILIDADE: Chave não encontrada {e}")


# ------------------------------------------------------------
# Tentativa de integração direta (VAI FALHAR)
# ------------------------------------------------------------
if __name__ == "__main__":
    print("\nTentando integrar Sistema RH com Sistema Financeiro...\n")
    processar_pagamento(funcionario_rh)
```

### Codigo-fonte: `legacy_systems.py`

```python
# lab3/legacy_systems.py
from integrador_esb import IntegradorESB

# ============================================================
# LABORATÓRIO 3 - PARTE 1
# DESAFIO DAS ILHAS DE DADOS
# ============================================================

"""
Este script demonstra um problema clássico de integração:
Dois sistemas legados que utilizam estruturas de dados diferentes
e não conseguem se comunicar diretamente.

Aqui ocorre um problema de INTEROPERABILIDADE TÉCNICA,
pois os formatos das chaves são incompatíveis.
"""

# ------------------------------------------------------------
# Sistema A: Ilha de Dados de RH
# ------------------------------------------------------------
# Estrutura utilizada pelo setor de Recursos Humanos
funcionario_rh = {
    "id_func": 101,
    "nome_completo": "Eduardo Amaral",
    "salario_base": 5000.00,
}


# ------------------------------------------------------------
# Sistema B: Ilha de Dados Financeira
# ------------------------------------------------------------
# Este sistema espera campos com nomes diferentes:
# 'cod' e 'valor'
def processar_pagamento(dados_pagamento):
    """
    Sistema financeiro que processa pagamentos.
    Ele falha se as chaves esperadas não existirem.
    """
    try:
        print(
            f"Processando ID {dados_pagamento['cod']}: Valor R$ {dados_pagamento['valor']}"
        )
    except KeyError as e:
        print(f"ERRO DE INTEROPERABILIDADE: Chave não encontrada {e}")


# ------------------------------------------------------------
# Tentativa de integração direta (VAI FALHAR)
# ------------------------------------------------------------
if __name__ == "__main__":
    print("\n1) Tentativa direta de integrar RH com Financeiro...\n")
    processar_pagamento(funcionario_rh)

    print("\n2) Integração usando o middleware/ESB...\n")
    dados_convertidos = IntegradorESB.transformar_rh_para_financeiro(funcionario_rh)
    processar_pagamento(dados_convertidos)

```

### Codigo-fonte: `integrador_esb.py`

...

```python
# lab3/microservico_eping.py
# ============================================================
# LABORATÓRIO 3 - PARTE 3
# API REST e Microserviço seguindo padrão ePING
# ============================================================

"""
Este script simula um Microserviço Governamental
seguindo o padrão ePING (Arquitetura de Interoperabilidade do Governo).

Aqui ocorre:

- INTEROPERABILIDADE TÉCNICA:
  Uso de HTTP + JSON como protocolo padrão.

- INTEROPERABILIDADE SEMÂNTICA:
  Estrutura padronizada dos dados (cpf, nome, status).

- MODULARIZAÇÃO:
  O serviço funciona de forma independente dos sistemas consumidores.
"""

from flask import Flask, jsonify

app = Flask(__name__)

# ------------------------------------------------------------
# Base de dados simulada (Contrato Padronizado)
# ------------------------------------------------------------
database = {
    1: {
        "cpf": "123.456.789-00",
        "nome": "Eduardo Amaral",
        "status": "Ativo"
    },
    2: {
        "cpf": "987.654.321-00",
        "nome": "Maria Silva",
        "status": "Inativo"
    }
}

# ------------------------------------------------------------
# Endpoint REST
# ------------------------------------------------------------
@app.route('/api/cidadao/<int:id>', methods=['GET'])
def get_cidadao(id):
    """
    Endpoint que retorna dados padronizados.
    O contrato da API garante interoperabilidade semântica.
    """
    cidadao = database.get(id, {"erro": "Não encontrado"})
    return jsonify(cidadao)

# simulaçao de um microserviço ePING que pode ser consumido por diversos sistemas
# curl http://localhost:5000/api/cidadao/1

# ------------------------------------------------------------
# Execução do Serviço
# ------------------------------------------------------------
if __name__ == '__main__':
    print("======================================")
    print("Serviço ePING rodando na porta 5000...")
    print("Acesse: http://localhost:5000/api/cidadao/1")
    print("======================================")
    app.run(port=5000)
```

### Codigo-fonte: `interop_semantica.py`

```python
# lab3/interop_semantica.py
# ============================================================
# LABORATÓRIO 3 - PARTE 4
# INTEROPERABILIDADE SEMÂNTICA
# ============================================================

"""
Problema:
O Sistema Internacional pode enviar temperatura em Fahrenheit.
O Sistema de Saúde trabalha apenas com Celsius.

Se enviarmos apenas o número, o significado pode ser interpretado errado.

Aqui ocorre INTEROPERABILIDADE SEMÂNTICA,
pois estamos traduzindo o SIGNIFICADO do dado.
"""

def tradutor_universal(valor, unidade_origem):
    """
    Converte temperatura para Celsius se necessário.
    Garante que o Sistema de Saúde sempre receba dados padronizados.
    """

    print("\n======================================")
    print(f"Dado recebido: {valor}°{unidade_origem}")
    print("======================================")

    if unidade_origem.upper() == "F":
        # Conversão de Fahrenheit para Celsius
        valor_convertido = (valor - 32) * 5/9
        print(f"Convertendo de Fahrenheit para Celsius...")
        print(f"Valor convertido: {valor_convertido:.2f}°C")
        return valor_convertido, "C"

    elif unidade_origem.upper() == "C":
        print("Nenhuma conversão necessária.")
        return valor, "C"

    else:
        raise ValueError("Unidade desconhecida! Use 'C' ou 'F'.")


# ------------------------------------------------------------
# Simulação de envio para Sistema de Saúde
# ------------------------------------------------------------
def enviar_para_sistema_saude(valor, unidade):
    print("\nEnviando para Sistema de Saúde...")
    print(f"Temperatura registrada: {valor:.2f}°{unidade}")
    print("Interoperabilidade semântica garantida.\n")


if __name__ == "__main__":

    # Cenário 1 — Dado já em Celsius
    valor1, unidade1 = tradutor_universal(37, "C")
    enviar_para_sistema_saude(valor1, unidade1)

    # Cenário 2 — Dado em Fahrenheit (precisa conversão)
    valor2, unidade2 = tradutor_universal(100, "F")
    enviar_para_sistema_saude(valor2, unidade2)
```

### Codigo-fonte: `padronizacao_eping.py`

```python
# lab3/padronizacao_eping.py
# ============================================================
# LABORATÓRIO 3 - PARTE 5
# GOVERNANÇA E PADRONIZAÇÃO (ePING)
# ============================================================

"""
Problema:
O Ministério da Fazenda usa CPF com pontos e traços.
O Ministério da Saúde exige CPF apenas numérico.

Sem padronização, os sistemas não conseguem cruzar dados.

Aqui ocorre INTEROPERABILIDADE ORGANIZACIONAL,
pois estamos aplicando um contrato institucional de dados.
"""

import re


def normalizar_cadastro(dados_origem):
    """
    Gateway de Integração (ESB Governamental).
    Garante que os dados sigam o contrato organizacional.
    """

    print("\n======================================")
    print("DADOS ORIGINAIS RECEBIDOS")
    print("======================================")
    print(dados_origem)

    # Remover caracteres especiais do CPF
    cpf_limpo = re.sub(r'\D', '', dados_origem["cpf"])

    # Nome em maiúsculas (padrão de sistemas legados)
    nome_padronizado = dados_origem["nome"].upper()

    dados_padronizados = {
        "cpf": cpf_limpo,
        "nome": nome_padronizado,
        "origem_validada": True
    }

    print("\n======================================")
    print("DADOS PADRONIZADOS (CONTRATO ePING)")
    print("======================================")
    print(dados_padronizados)

    return dados_padronizados


# ------------------------------------------------------------
# Simulação de integração entre Ministérios
# ------------------------------------------------------------
if __name__ == "__main__":

    # Dados vindos da Fazenda
    dados_fazenda = {
        "cpf": "123.456.789-00",
        "nome": "Eduardo Amaral"
    }

    dados_normalizados = normalizar_cadastro(dados_fazenda)
```

## 9. Evidencias de execucao dos scripts

As proximas execucoes registram as saidas dos scripts principais.
O microservico Flask foi validado na Parte 3 por meio de
`test_client()`, evitando iniciar um servidor permanente durante a
geracao do relatorio.

In [8]:
def executar_script(nome_arquivo: str) -> None:
    resultado = subprocess.run(
        [sys.executable, str(BASE_DIR / nome_arquivo)],
        capture_output=True,
        text=True,
        check=False,
    )
    print("=" * 70)
    print(f"$ python {nome_arquivo}")
    print("=" * 70)
    print(resultado.stdout)
    if resultado.stderr:
        print("STDERR:")
        print(resultado.stderr)
    print(f"Codigo de retorno: {resultado.returncode}")


for script in [
    "legacy_systems_original.py",
    "legacy_systems.py",
    "interop_semantica.py",
    "padronizacao_eping.py",
]:
    executar_script(script)

$ python legacy_systems_original.py

Tentando integrar Sistema RH com Sistema Financeiro...

ERRO DE INTEROPERABILIDADE: Chave não encontrada 'cod'

Codigo de retorno: 0
$ python legacy_systems.py

1) Tentativa direta de integrar RH com Financeiro...

ERRO DE INTEROPERABILIDADE: Chave não encontrada 'cod'

2) Integração usando o middleware/ESB...

Processando ID 101: Valor R$ 5000.0

Codigo de retorno: 0


$ python interop_semantica.py

Dado recebido: 37°C
Nenhuma conversão necessária.

Enviando para Sistema de Saúde...
Temperatura registrada: 37.00°C
Interoperabilidade semântica garantida.


Dado recebido: 100°F
Convertendo de Fahrenheit para Celsius...
Valor convertido: 37.78°C

Enviando para Sistema de Saúde...
Temperatura registrada: 37.78°C
Interoperabilidade semântica garantida.


Codigo de retorno: 0
$ python padronizacao_eping.py

DADOS ORIGINAIS RECEBIDOS
{'cpf': '123.456.789-00', 'nome': 'Eduardo Amaral'}

DADOS PADRONIZADOS (CONTRATO ePING)
{'cpf': '12345678900', 'nome': 'EDUARDO AMARAL', 'origem_validada': True}

Codigo de retorno: 0


## 10. Conclusao

O laboratorio demonstra que interoperabilidade nao e apenas conectar
sistemas. A integracao direta falha quando os contratos de dados sao
diferentes. O ESB resolve a transformacao entre sistemas legados; a
API REST cria um contrato tecnico padronizado; o tradutor de
unidades preserva o significado dos dados; e o gateway de
padronizacao centraliza regras organizacionais. Em conjunto, essas
solucoes reduzem acoplamento, aumentam governanca e tornam a
arquitetura mais preparada para mudancas.